---
authors:
- ktyle
---

# METAR GeoParquet Interactive Visualization

## Overview
   
Within this notebook, we will create interactive visualizations of METAR data at a single hour. We will use the following libraries for our visualizations:

1. [Geopandas](https://geopandas.org/en/stable/)

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Cartopy Intro](https://foundations.projectpythia.org/core/cartopy/cartopy.html) | Required | Projections and Features |
| [Pandas](https://foundations.projectpythia.org/core/pandas.html) | Required | Tabular Datasets |

- **Time to learn**: 20 minutes
---

## Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
import geopandas as gpd
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
import duckdb
import folium

### By default, set the date and time to one hour prior to the current time. Or, specify a past date and hour.

In [ ]:
# Use the current time, or set your own for a past time.
# Set current to False if you want to specify a past time.

nowTime = datetime.now()

current = True
#current = False
if (current):
    time_1 = datetime.now()
    offset = timedelta(hours = 1) # Worldwide METARS typically don't come in until ~15 minutes past the hour
    time_1 = time_1 - offset
else:
    year = 2026
    month = 1
    day = 1
    hour = 0
    time_1 = datetime(year, month, day, hour)

The timestamps of many hourly METAR reports are 5-10 minutes prior to the top of the hour. When we make our DuckDB query, we will specify a beginning and end time, one hour apart.

In [ ]:
time_0 = time_1 - timedelta(hours=1)
YYYY_0 = time_0.strftime("%Y")
YYYY_1 = time_1.strftime("%Y")
date_string = time_1.strftime("%Y-%m-%d %H00 UTC")
print(time_0, time_1)
# Handle edge case when the two hours straddle the end/beginning of a yearw
if (YYYY_0 == YYYY_1):
    URLs = [f'https://data.source.coop/dynamical/asos-parquet/year={YYYY_1}/data.parquet']
else:
    URLs = [f'https://data.source.coop/dynamical/asos-parquet/year={YYYY_0}/data.parquet',
            f'https://data.source.coop/dynamical/asos-parquet/year={YYYY_1}/data.parquet']


In [ ]:
date_string

In [ ]:
URLs

### Open the GeoParquet file for the specific year.

In [ ]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    WHERE 
---      country = 'FR' AND
---    station = 'ENGM' AND
    valid BETWEEN $2 AND $3
    ORDER BY country
""", [URLs, time_0, time_1]).fetchdf()

In [ ]:
df

In [ ]:
gdf = gpd.GeoDataFrame(df,geometry=gpd.points_from_xy(df.longitude,df.latitude))

In [ ]:
gdf.explore()

:::{warning}Where are the cartographic features?
Well … we have an interactive frame … and we can infer worldwide geography from it  … but where are the cartographic features??

We still have a little more work to do:

While the points certainly look like latitude and longitudes, we need to explicitly assign a projection to the Dataframe before we can view it on a map. One way is to assign a coordinate reference system code, via [EPSG](https://epsg.io) … in this case, EPSG 4326.

Note the arguments to `set_crs`:

`epsg = 4326`: Assign the specified CRS

`inplace = True`: The gdf object is updated without the need to assign a new dataframe object

`allow_override` = True: If a CRS had previously been applied, override with the EPSG value specified.
:::

In [ ]:
gdf.set_crs(epsg=4326, inplace=True, allow_override=True)

In [ ]:
gdf.explore()

:::{error}Missing data from many countries!
It seems that many countries don't seem to have any METAR observations (e.g., Norway, Nigeria). They definitely do exist, but the seem to not be included in this archive!
:::

Now, let's choose a specific variable ... in this case, `tmpc` (2-meter temperature in °C)

In [ ]:
gdf.explore(column='tmpc')

:::{note}
The color scale is unaffected by any location whose temperature value was missing ... GeoPandas "knows" to exlucde `NaN` from the range of valid values.
:::

Plot hourly precipitation

In [ ]:
gdf.explore(column='p01m')

:::{note}
With few exceptions, hourly precip is only provided by US stations. Any value less than 0.01 is denoted as a *trace*. Let's drop all values < 0.01. 
:::

In [ ]:
gdf.loc[gdf['p01m'] < 0.01, 'p01m'] = np.nan

In [ ]:
gdf = gdf.dropna(subset=['p01m']).reset_index(drop=True)

In [ ]:
gdf.explore(column='p01m')

### To-do: add title to the graphics

In [ ]:
m = gdf.explore()

# Add title
title_string = f"Hourly Precip (mm), {date_string}"

title_html = f"""
<h3 align="center" style="font-size:20px">
    <b>{title_string}</b>
</h3>
"""

m.get_root().html.add_child(folium.Element(title_html))

# Save or display
#m.save("map.html")

In [ ]:
m

## References

1. [GeoPandas](https://geopandas.org)
1. [EPSG](https://epsg.io)

## What's next?
We will use [Lonboard](https://developmentseed.org/lonboard/latest/) as an additional interactive visualization tool.